# verify — Phi-4 Multimodal (HuggingFace Transformers)

Runs **microsoft/Phi-4-multimodal-instruct** locally inside Colab to answer questions from the main `questions.json` dataset.

**Data setup:** Mount your Google Drive and point `DRIVE_ROOT` to the `QuestionBank/` folder, which must contain:
```
Data/
  questions.json
  questions/   ← .txt files with question text
  images/      ← image files referenced in questions.json
```

Answers are written to `/content/outputs/verify/<id>.txt` and optionally copied back to Drive.

> **Runtime:** A100 GPU (Colab Pro) strongly recommended — model is ~14B parameters.

In [ ]:
!pip install -q transformers accelerate pillow torch torchvision

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Paste your HuggingFace token: ").strip()

login(token=hf_token)
print("Logged in to HuggingFace.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"

print("Loading processor…")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (may take a few minutes)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

In [ ]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount("/content/drive")

# ── Set this to where QuestionBank/ lives on your Drive ──────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/QuestionBank"
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR  = os.path.join(DRIVE_ROOT, "Data")
JSON_FILE = os.path.join(DATA_DIR, "questions.json")

assert os.path.isfile(JSON_FILE), f"Not found: {JSON_FILE}"
print("Found questions.json ✓")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/content/outputs/verify"
MAX_NEW_TOKENS  = 1024

SYSTEM_PROMPT = (
    "You are expert computer science tutor. "
    "Answer the given question step by step. "
    "Begin by explaining your reasoning process clearly. "
    "Think step by step before answering the question."
)

FILTER_IDS = None   # e.g. ["1", "5", "12"]
LAST_N     = None   # e.g. 10

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json
from PIL import Image

def load_entries():
    with open(JSON_FILE, encoding="utf-8") as f:
        return json.load(f)

def read_question_file(rel_path):
    full = os.path.join(DATA_DIR, rel_path.lstrip("./"))
    if not os.path.exists(full):
        return ""
    with open(full, encoding="utf-8") as f:
        return f.read().strip()

def resize_to_224(img):
    w, h   = img.size
    side   = max(w, h)
    padded = Image.new("RGB", (side, side), (255, 255, 255))
    padded.paste(img, ((side - w) // 2, (side - h) // 2))
    return padded.resize((224, 224), Image.LANCZOS)

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    question_text = read_question_file(entry["question"])
    if not question_text:
        print(f"  [SKIP] #{entry['id']} — question file missing")
        return

    pil_images = []
    for rel in entry.get("images", []):
        full = os.path.join(DATA_DIR, rel.lstrip("./"))
        if os.path.exists(full):
            pil_images.append(resize_to_224(Image.open(full).convert("RGB")))

    image_tags  = "".join(f"<|image_{i+1}|>" for i in range(len(pil_images)))
    phi4_prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{image_tags}{question_text}<|end|>\n"
        f"<|assistant|>\n"
    )

    inputs = processor(
        text=phi4_prompt,
        images=pil_images if pil_images else None,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer     = processor.decode(new_tokens, skip_special_tokens=True).strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)
    print(f"  [OK]   #{entry['id']} — {entry.get('topic', '')} → {out_path}")

print("Helpers defined.")

In [ ]:
all_entries = load_entries()

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

print(f"Model  : {MODEL_ID}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
for entry in entries:
    try:
        ask(entry)
    except Exception as exc:
        import traceback
        print(f"  [ERR]  #{entry['id']} — {type(exc).__name__}: {exc}")
        traceback.print_exc()

print("\nDone.")

In [ ]:
# Copy outputs back to Drive (optional)
import shutil
dest = os.path.join(DRIVE_ROOT, "verify", "phi-4-multimodal")
os.makedirs(dest, exist_ok=True)
for f in os.listdir(OUTPUT_DIR):
    shutil.copy(os.path.join(OUTPUT_DIR, f), os.path.join(dest, f))
print(f"Outputs copied to Drive: {dest}")